In [1]:
!pip install -q pandas sentence-transformers scikit-learn

In [2]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
data = {
    "Question": [
        "What are the clinic working hours?",
        "How can I book an appointment?",
        "What is the consultation fee?",
        "Does the clinic provide blood tests?",
        "Is an appointment required?",
        "Can I cancel my appointment?",
        "Does the clinic have a pharmacy?",
        "Does the clinic provide emergency services?",
        "Which doctors are available?",
        "How can I contact the clinic?"
    ],

    "Answer": [
        "The clinic is open Monday to Saturday from 9:00 AM to 6:00 PM.",
        "You can book an appointment by calling the clinic reception or visiting the appointment desk.",
        "The general consultation fee is ₹500.",
        "Yes, basic blood tests are available at the clinic laboratory.",
        "Appointments are recommended, but walk-in consultations are available depending on doctor availability.",
        "Yes. Please contact the reception at least 2 hours before your appointment.",
        "Yes, the clinic has an in-house pharmacy for commonly prescribed medicines.",
        "The clinic provides basic emergency assistance. For serious emergencies, contact your nearest emergency hospital.",
        "The clinic has general physicians, pediatricians and dermatology specialists.",
        "You can contact the clinic reception during working hours for appointments and enquiries."
    ]
}

df = pd.DataFrame(data)

df

,Question,Answer
0,What are the clinic working hours?,The clinic is open Monday to Saturday from 9:0...
1,How can I book an appointment?,You can book an appointment by calling the cli...
2,What is the consultation fee?,The general consultation fee is ₹500.
3,Does the clinic provide blood tests?,"Yes, basic blood tests are available at the cl..."
4,Is an appointment required?,"Appointments are recommended, but walk-in cons..."
5,Can I cancel my appointment?,Yes. Please contact the reception at least 2 h...
6,Does the clinic have a pharmacy?,"Yes, the clinic has an in-house pharmacy for c..."
7,Does the clinic provide emergency services?,The clinic provides basic emergency assistance...
8,Which doctors are available?,"The clinic has general physicians, pediatricia..."
9,How can I contact the clinic?,You can contact the clinic reception during wo...


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [5]:
faq_questions = df["Question"].tolist()

faq_embeddings = model.encode(
    faq_questions,
    convert_to_numpy=True
)

print("FAQ embeddings created!")
print("Embedding shape:", faq_embeddings.shape)

FAQ embeddings created!
Embedding shape: (10, 384)


In [7]:
def retrieve_answer(user_question, threshold=0.35):

    # Convert user question into embedding
    query_embedding = model.encode(
        [user_question],
        convert_to_numpy=True
    )

    # Calculate similarity
    similarity_scores = cosine_similarity(
        query_embedding,
        faq_embeddings
    )[0]

    # Find most relevant FAQ
    best_index = np.argmax(similarity_scores)
    best_score = similarity_scores[best_index]

    # Check similarity threshold
    if best_score >= threshold:

        return {
            "question": df.iloc[best_index]["Question"],
            "answer": df.iloc[best_index]["Answer"],
            "score": best_score
        }

    else:
        return {
            "question": None,
            "answer": "Sorry, I could not find a relevant answer in the clinic knowledge base.",
            "score": best_score
        }

In [8]:
question = "What time does the clinic open?"

result = retrieve_answer(question)

print("User Question:", question)
print("\nRetrieved FAQ:", result["question"])
print("\nChatbot Answer:", result["answer"])
print("\nSimilarity Score:", round(result["score"], 2))

User Question: What time does the clinic open?

Retrieved FAQ: What are the clinic working hours?

Chatbot Answer: The clinic is open Monday to Saturday from 9:00 AM to 6:00 PM.

Similarity Score: 0.77


In [9]:
question = "How much do I need to pay for consultation?"

result = retrieve_answer(question)

print("User Question:", question)
print("\nRetrieved FAQ:", result["question"])
print("\nChatbot Answer:", result["answer"])
print("\nSimilarity Score:", round(result["score"], 2))

User Question: How much do I need to pay for consultation?

Retrieved FAQ: What is the consultation fee?

Chatbot Answer: The general consultation fee is ₹500.

Similarity Score: 0.87


In [10]:
question = "Where can I buy a laptop?"

result = retrieve_answer(question)

print("User Question:", question)
print("\nRetrieved FAQ:", result["question"])
print("\nChatbot Answer:", result["answer"])
print("\nSimilarity Score:", round(result["score"], 2))

User Question: Where can I buy a laptop?

Retrieved FAQ: None

Chatbot Answer: Sorry, I could not find a relevant answer in the clinic knowledge base.

Similarity Score: 0.18


In [ ]:
print("🏥 Clinic FAQ RAG Chatbot")
print("Type 'exit' to stop.\n")

while True:

    user_question = input("You: ")

    if user_question.lower() == "exit":
        print("Chatbot: Thank you!")
        break

    result = retrieve_answer(user_question)

    print("\nChatbot:", result["answer"])
    print("Similarity:", round(result["score"], 2))
    print()

🏥 Clinic FAQ RAG Chatbot
Type 'exit' to stop.


Chatbot: The general consultation fee is ₹500.
Similarity: 0.98


Chatbot: You can book an appointment by calling the clinic reception or visiting the appointment desk.
Similarity: 0.99


Chatbot: Yes, basic blood tests are available at the clinic laboratory.
Similarity: 0.99


Chatbot: Sorry, I could not find a relevant answer in the clinic knowledge base.
Similarity: 0.18

